# Spam SMS-ek (30 pont)

**A notebookot importáld be a Colab rendszerbe, majd abban dolgozz!**

**Versenyző neve:**

Intelligens János egyik nap arra ébredt, hogy ezernél is több SMS áraszotta el a régi Nokia 3310-esét. Azt szeretné megtudni, hogy mely SMS-ekkel kell foglalkoznia és melyek azok, amelyek spamnek minősülnek.
Segíts neki eldönteni, mely SMS-ek minősülnek spamnek!

---
# Előkészítések

**Útmutatók a feladat megoldásához szükséges eszközökhöz:**
1. [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
2. [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
3. [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
4. [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
5. [PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)
6. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
7. [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)

A megoldáshoz szükséges **SMS-ek** az [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) adathalmazból származnak.

Alternatív forrás: [Kaggle SMS Spam Collection](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

⚠️ **A modell súlyok (`pth` fájl) már nem elérhetőek.** A 4. feladat (súlyok betöltése) és az 5. feladat (modell tesztelése) ezek nélkül nem teljesíthető teljesen.

In [ ]:
# Az SMS adathalmaz letöltése (UCI SMS Spam Collection)
# Eredeti Google Drive link már nem elérhető.
# Alternatíva: tölts le manuálisan a Kaggle-ről:
# https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

# Kaggle CLI-vel:
# !pip install -q kaggle
# !kaggle datasets download -d uciml/sms-spam-collection-dataset
# !unzip -qq sms-spam-collection-dataset.zip

# ⚠️ A modell súlyok már nem elérhetőek.
# !gdown -qq 1-MtzrY1JqJEfvxuSASobGOYWAoZ3NeGu  # SMS adat (nem elérhető)
# !gdown -qq 1I7IJZ-ieEXCp-AVmrUzaFXPjLCNNLvKc  # Modell súlyok (nem elérhető)

## Szükséges Könyvtárak

Importáltunk néhány könyvtárat a kezdéshez, de nyugodtan használhatsz bármilyen PyTorch-alapú eszközt, ha szükséges. Kérjük, vedd figyelembe, hogy a Keras és a TensorFlow **NEM ENGEDÉLYEZETT** ennek a feladatnak a megoldásához!


In [ ]:
import torch
import re
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

## Az SMS Adathalmaz szemléltetése

Az alábbi kód vizualizálja az adathalmaz számosságát és az első öt beérkezett SMS-t.

Minden szöveghez tartozik egy besorolás is, ami arrra vonatkozik, hogy az adott SMS **spam** vagy **legit** (nem spam). [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

In [ ]:
df = pd.read_csv('./sms.csv')
print("Beérkezett SMS-ek száma", len(df))
df.head()

# 1. Feladat: Exploratív Adatelemzés (8 Pont)

Az első feladatban az SMS adathalmazon szeretnénk kezdeti elemzést és adattisztítást végezni, hogy jobban megismerjük a János telefonjára érkezett üzeneteket. A feladataid a következők:

1. Számold össze hány egyedi szó (token) található az adathalmazban! (1 pont)

2. Számítsd ki a **spam** és **legit** kategóriába sorolt sms-ek számát! (1 pont)

3. Tisztítsd meg az adathalmazt a megadott clean_text függvény segítségével! A függvényt ne módosítsd! (1 pont)

4. A tisztított adatokat új Pandas oszlopként illeszd hozzá az eredeti DataFrame-hez! Vizualizáld a frissített adathalmazt! (1 pont)

[Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)

In [ ]:
def clean_text(text):
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower()

def unique_word_count():
    raise NotImplementedError("This function has not been implemented yet.")

def sentiment_count():
    raise NotImplementedError("This function has not been implemented yet.")

# 2. Feladat: Adatok előkészítése (10 Pont)

A második feladatban egy szótárat szeretnénk készíteni, ami minden egyes egyedi szóhoz egy számszerű értéket rendel. Ezt a folyamatot **Tokenizálás**nak hívjuk, melynek a fenti módszer az egyik legegyszerűbb formája. A feladatok megoldására akármilyen könyvtárat használhatsz! A feladataid a következők:

1. Készíts egy saját Tokenizálót, ami minden szóhoz egy egyedi indexet rendel! (2 pont) [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
2. A Tokenizálót futtasd le a teljes adatállományon, hogy minden SMS-hez egy számsorozat reprezentációt rendelj! (1 pont)
3. Nézd meg, hogy melyik SMS a leghosszabb, majd ahhoz igazítva üres tokenekkel töltsd fel az összes többi SMS-t, hogy egyező hosszúsági Token szekvenciákat kapj! (2 pont) [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
4. Az SMS-ek besorolásait tárold el bináris kódolással: `[spam = 0, legit = 1]` (1 pont)
5. A kapott token szekvenciákat és bináris kódolású SMS besorolásokat add hozzá a kezdetleges Pandas DataFrame-hez (1 pont)

In [ ]:
def tokenize():
    raise NotImplementedError("This function has not been implemented yet.")

def build_vocab():
    raise NotImplementedError("This function has not been implemented yet.")

def pad_token_sequences():
    raise NotImplementedError("This function has not been implemented yet.")

def binary_encoding():
    raise NotImplementedError("This function has not been implemented yet.")

# 3. Feladat: PyTorch Dataset készítése (3 Pont)

A harmadik feladathoz egy PyTorch Dataset-et kell létrehoznod, amely képes a token szekvenciákat és a hozzájuk tartozó bináris besorolást kezelni. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, sequences, labels):
        raise NotImplementedError("This function has not been implemented yet.")

    def __len__(self):
        raise NotImplementedError("This function has not been implemented yet.")

    def __getitem__(self, idx):
        raise NotImplementedError("This function has not been implemented yet.")

# 4. Feladat: Modell felépítése (5 Pont)

A következő ábra alapján készítsd el az SMS-ek klasszifikációjához szükséges neurális háló modelljét! Ez a modell képes lesz a tokenizált szövegek feldolgozására majd osztályozni őket a spam és legit kategóriába.

1. A modell létrehozása után töltsd be a súlyokat a modellbe!

<a href="https://ibb.co/F6KF3xC"><img src="https://i.ibb.co/b1PkmWD/net-emb.png" alt="net-emb" border="0"></a>

A `vocab size` az egyedi tokenek száma (9476 + PAD token = 9477), az `output_dim` pedig a lehetséges osztályok száma (`spam`, `legit`). (4 pont)

[Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)

2. Töltsd be a modellsúlyokat az elkészített modellbe a `jános_email_modell.pth` nevű modellfájl segítségével! (1 pont) ([Pytorch Loading Weights](https://pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html#:~:text=To%20load%20model%20weights%2C%20you,parameters%20using%20load_state_dict()%20method.&text=be%20sure%20to%20call%20model,normalization%20layers%20to%20evaluation%20mode))

In [ ]:
EMBEDDING_DIM = 16

class SMSModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super(SMSModel, self).__init__()
        raise NotImplementedError("This function has not been implemented yet.")

    def forward(self, x):
        raise NotImplementedError("This function has not been implemented yet.")


def load_weights():
    raise NotImplementedError("This function has not been implemented yet.")

# 5. Feladat: Modell tesztelése (4 Pont)

János telefonjára ebben a pillanatban a következő SMS érkezett be:

`Hey, János. I have a crazy hiking opportunity for you in the best place on Earth, the Himalayas. I can be your hiking partner right away, just send me your bank information and personal details, and let's get started.`

1. Segíts neki eldönteni, hogy a kapott üzenet spam üzenet vagy sem! (2 pont)
2. Küldj üzenetet Jánosnak, próbálj olyan üzenetet írni, amit rosszindulatú szándékkal hozol létre (`spam`), viszont a modell valósnak (`legit`) fogja osztályozni! (2 pont)

[PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)

In [ ]:
def predict():
    raise NotImplementedError("This function has not been implemented yet.")

---
**A feladat végére értél**. A kész Notebook-ot töltsd le a következő módon:
```
File → Download → Download .ipynb
```
majd a **többi megoldással együtt** töltsd fel a CMS rendszerbe becsomagolva **(.zip)**.